# KNN: from scratch vs scikit-learn

Hyperparameters (s and distance) choose based on **validation** set.

In [1]:
import os
import pickle
import sys
import time

import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

NOTEBOOK_DIR = os.path.abspath('.')
SRC_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', 'src'))
SPLIT_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', 'data'))
MODEL_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', 'models'))
os.makedirs(MODEL_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)

from knn import KNN
from evaluate import evaluate, format_report

In [2]:
LABEL_MAP = {'Dropout': 0, 'Enrolled': 1, 'Graduate': 2}

def load_split(name):
    frame = pd.read_csv(os.path.join(SPLIT_DIR, name))
    X = frame.drop(columns=['Student_ID', 'Target']).to_numpy()
    y = frame['Target'].map(LABEL_MAP).to_numpy()
    return X, y

X_train, y_train = load_split('train_preprocessed.csv')
X_val, y_val = load_split('validation_preprocessed.csv')
X_test, y_test = load_split('test_preprocessed.csv')

print('train', X_train.shape, 'val', X_val.shape, 'test', X_test.shape)
print('classes in train:', np.unique(y_train))

train (1857, 211) val (619, 211) test (620, 211)
classes in train: [0 1 2]


## 1. Choose k and distance metric on the validation set

In [3]:
def make_model(implementation, k, metric):
    if implementation == 'scratch':
        return KNN(k=k, metric=metric)
    return KNeighborsClassifier(n_neighbors=k, metric=metric)

K_VALUES = [1, 3, 5, 7, 9, 11, 15, 21, 31]
METRICS = ['euclidean', 'manhattan']

rows = []
for implementation in ['scratch', 'sklearn']:
    for metric in METRICS:
        for k in K_VALUES:
            model = make_model(implementation, k, metric)
            model.fit(X_train, y_train)
            predictions = model.predict(X_val)
            result = evaluate(y_val, predictions, 0, 0, classes=[0, 1, 2])
            rows.append({
                'implementation': implementation,
                'metric': metric,
                'k': k,
                'val_accuracy': round(result['accuracy'], 4),
                'val_macro_f1': round(result['macro_f1'], 4),
            })

grid = pd.DataFrame(rows)
grid.pivot_table(index='k', columns=['implementation', 'metric'], values='val_macro_f1')

implementation   scratch             sklearn          
metric         euclidean manhattan euclidean manhattan
k                                                     
1                 0.5703    0.5772    0.5703    0.5772
3                 0.5735    0.6088    0.5735    0.6088
5                 0.5953    0.6161    0.5953    0.6161
7                 0.5921    0.6076    0.5921    0.6076
9                 0.5791    0.5956    0.5791    0.5956
11                0.5924    0.5941    0.5924    0.5941
15                0.5852    0.5764    0.5852    0.5764
21                0.5426    0.5884    0.5426    0.5884
31                0.5512    0.5331    0.5512    0.5331

In [4]:
best = {}
for implementation in ['scratch', 'sklearn']:
    part = grid[grid['implementation'] == implementation]
    best[implementation] = part.loc[part['val_macro_f1'].idxmax()]
    print(implementation, '-> best k =', best[implementation]['k'],
          ', metric =', best[implementation]['metric'],
          ', val macro F1 =', best[implementation]['val_macro_f1'],
          ', val accuracy =', best[implementation]['val_accuracy'])

scratch -> best k = 5 , metric = manhattan , val macro F1 = 0.6161 , val accuracy = 0.7141
sklearn -> best k = 5 , metric = manhattan , val macro F1 = 0.6161 , val accuracy = 0.7141


## 2. Final evaluation on the test set

In [5]:
def measure(model, name, X_eval, y_eval):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    predictions = model.predict(X_eval)
    inference_seconds = time.perf_counter() - start
    results = evaluate(y_eval, predictions, fit_seconds, inference_seconds, classes=[0, 1, 2])
    results['name'] = name
    results['model'] = model
    results['predictions'] = predictions
    return results

runs = []
for implementation in ['scratch', 'sklearn']:
    chosen = best[implementation]
    model = make_model(implementation, int(chosen['k']), chosen['metric'])
    name = 'KNN ' + implementation + ' (k=' + str(int(chosen['k'])) + ', ' + chosen['metric'] + ')'
    runs.append(measure(model, name, X_test, y_test))

for run in runs:
    print(run['name'])
    print(format_report(run))
    print()

KNN scratch (k=5, manhattan)
accuracy    0.7177
macro F1    0.6378
fit time    0.00 s
infer time  1.534 s

class         precision  recall   f1
0            0.7527     0.6884   0.7192
1            0.4302     0.3333   0.3756
2            0.7699     0.8742   0.8187

confusion matrix, rows true, columns predicted
                       0          1          2
0                    137         28         34
1                     27         37         47
2                     18         21        271

KNN sklearn (k=5, manhattan)
accuracy    0.7177
macro F1    0.6378
fit time    0.00 s
infer time  0.060 s

class         precision  recall   f1
0            0.7527     0.6884   0.7192
1            0.4302     0.3333   0.3756
2            0.7699     0.8742   0.8187

confusion matrix, rows true, columns predicted
                       0          1          2
0                    137         28         34
1                     27         37         47
2                     18         21        271

In [6]:
pd.DataFrame([
    {
        'model': run['name'],
        'accuracy': round(run['accuracy'], 4),
        'macro_f1': round(run['macro_f1'], 4),
        'fit_s': round(run['fit_seconds'], 4),
        'infer_s': round(run['inference_seconds'], 3),
    }
    for run in runs
])

,model,accuracy,macro_f1,fit_s,infer_s
0,"KNN scratch (k=5, manhattan)",0.7177,0.6378,0.0001,1.534
1,"KNN sklearn (k=5, manhattan)",0.7177,0.6378,0.0020,0.060


scikit-learn is run with the same k and metric as the scratch model. If the scratch code is correct, the predictions should be (almost) identical. *A tiny difference can happen when two neighbours have exactly the same distance.

In [7]:
scratch_run = runs[0]
same_setting = KNeighborsClassifier(
    n_neighbors=int(best['scratch']['k']), metric=best['scratch']['metric']
).fit(X_train, y_train)
sklearn_predictions = same_setting.predict(X_test)
print('share of test rows with the same prediction:',
      round(float(np.mean(scratch_run['predictions'] == sklearn_predictions)), 4))

share of test rows with the same prediction: 1.0


## 3. Save and load

The scratch model uses its own `save` / `load` methods. The scikit-learn model is saved with pickle, like the other notebooks.

In [8]:
scratch_model = runs[0]['model']
scratch_path = os.path.join(MODEL_DIR, 'knn_scratch.pkl')
scratch_model.save(scratch_path)
loaded_scratch = KNN.load(scratch_path)

sklearn_model = runs[1]['model']
sklearn_path = os.path.join(MODEL_DIR, 'knn_sklearn.pkl')
with open(sklearn_path, 'wb') as file:
    pickle.dump(sklearn_model, file)
with open(sklearn_path, 'rb') as file:
    loaded_sklearn = pickle.load(file)

print('scratch: identical predictions after load:',
      bool(np.array_equal(scratch_model.predict(X_test), loaded_scratch.predict(X_test))))
print('sklearn: identical predictions after load:',
      bool(np.array_equal(sklearn_model.predict(X_test), loaded_sklearn.predict(X_test))))

scratch: identical predictions after load: True
sklearn: identical predictions after load: True
